In [1]:
!pip install transformers datasets -q

In [2]:
import pandas as pd
import numpy as np
import re
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util as st_util
from IPython.display import display, HTML
from google.colab import drive

In [3]:
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/Licenta/medical_dataset_final.csv'

Mounted at /content/drive


In [4]:
df = pd.read_csv(CSV_PATH, sep=';', on_bad_lines='skip')
df = df[['filename', 'rezumat', 'keywords']].copy()
df = df.dropna(subset=['rezumat', 'keywords'])
df['rezumat']  = df['rezumat'].str.strip()
df['keywords'] = df['keywords'].str.strip()

print(f"Total articole: {len(df)}")

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Train : {len(train_df)} articole")
print(f"Val   : {len(val_df)} articole")
print(f"Test  : {len(test_df)} articole")

Total articole: 1114
Train : 779 articole
Val   : 167 articole
Test  : 168 articole


In [5]:
MODEL_NAME = 'readerbench/RoGPT2-medium'

print(f"\nSe încarcă tokenizer-ul {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT2 nu are pad token implicit — îl setăm manual
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 512  # lungime totală input + output

PROMPT_TEMPLATE = "Rezumat: {rezumat} ### Cuvinte cheie: {keywords}"
INFERENCE_TEMPLATE = "Rezumat: {rezumat} ### Cuvinte cheie:"

def format_train(examples):
    texts = [
        PROMPT_TEMPLATE.format(rezumat=r, keywords=k) + tokenizer.eos_token
        for r, k in zip(examples['rezumat'], examples['keywords'])
    ]
    tokenized = tokenizer(
        texts,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

train_dataset = Dataset.from_pandas(train_df[['rezumat', 'keywords']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['rezumat', 'keywords']].reset_index(drop=True))

print("Tokenizez datele...")
train_dataset = train_dataset.map(format_train, batched=True, remove_columns=['rezumat', 'keywords'])
val_dataset   = val_dataset.map(format_train,   batched=True, remove_columns=['rezumat', 'keywords'])
print("Tokenizare completă.")


Se încarcă tokenizer-ul readerbench/RoGPT2-medium...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/869 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/985k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

Tokenizez datele...


Map:   0%|          | 0/779 [00:00<?, ? examples/s]

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Tokenizare completă.


In [6]:
print(f"\nSe încarcă modelul {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Parametri model: {model.num_parameters():,}")

# Data collator pentru language modeling cauzal
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # nu masked LM, ci cauzal LM (GPT-style)
)



Se încarcă modelul readerbench/RoGPT2-medium...


pytorch_model.bin:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: readerbench/RoGPT2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parametri model: 354,823,168


In [7]:
training_args = TrainingArguments(
    output_dir='/content/rogpt2_tmp',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=5e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=50,
    save_total_limit=2,
    fp16=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [8]:
print("\nÎncepe antrenarea...")
trainer.train()
print("Antrenare completă!")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.



Începe antrenarea...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.676909,2.551865
2,2.411164,2.519865
3,2.125999,2.535205
4,1.980798,2.587326
5,1.846357,2.633584


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Antrenare completă!


In [9]:
SAVE_PATH = '/content/drive/MyDrive/Licenta/rogpt2_model'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model salvat în {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model salvat în /content/drive/MyDrive/Licenta/rogpt2_model


In [10]:
print("\nGenerez keywords pentru setul de test...")

def generate_keywords(rezumat, max_new_tokens=60):
    prompt = INFERENCE_TEMPLATE.format(rezumat=rezumat)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=MAX_LEN - max_new_tokens,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decodăm doar tokenurile noi (fără promptul de input)
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Oprim la primul newline sau EOS dacă mai sunt caractere extra
    generated = generated.split('\n')[0].strip()
    return generated


Generez keywords pentru setul de test...


In [11]:
model.eval()
results = []
for _, row in test_df.iterrows():
    generated = generate_keywords(row['rezumat'])
    results.append({
        'filename'           : row['filename'],
        'keywords_generated' : generated,
        'keywords_original'  : row['keywords'],
    })

results_df = pd.DataFrame(results)

# Preview rapid
print("\nSample output:")
for _, r in results_df.head(3).iterrows():
    print(f"\n  Fișier    : {r['filename']}")
    print(f"  Generate  : {r['keywords_generated']}")
    print(f"  Originale : {r['keywords_original']}")


Sample output:

  Fișier    : RJP_2014_4_RO_Art-05.pdf
  Generate  : osmopenie; os; copil; adolescent; nou-născut; adult; vârstnic; vârstă; patologie; metabolism; vitamina d; calciu; paraproteinemie; hipercalciurie; hipopotasemie; deficit de vitamină D; hipospadias
  Originale : osteopenie; copil; diagnostic; tratament

  Fișier    : RJP_2008_3_Art-03.pdf
  Generate  : infecția congenitală; copil; nou-născut; sugar; mamă infectată; infecție; sarcină; anticorpi; sistem imunitar; imunitate; sindrom; virus; viral; parazitologie; vaccin; antigen; imunoglobuline; limfocite; celule imune;
  Originale : sindromul torch; infecții congenitale perinatale

  Fișier    : RJP_2008_2_Art-10.pdf
  Generate  : durere; membre inferioare; copil; vârstă; diagnostic; tratament; simptomatologie; evoluție; complicații; factori de risc pentru apariția durerii; metode de diagnostic și tratament ale durerii la copii; criterii de includere a copilului în programele de sănătate publică; recomandări de investiga

In [12]:
!pip install sentence-transformers -q

SIM_THRESHOLD = 0.6
print("\nSe încarcă modelul semantic...")
sem_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def parse_keywords(text):
    if pd.isna(text) or str(text).strip() == '':
        return []
    return [w.strip().lower() for w in re.split(r'[;,]', str(text)) if w.strip()]

def compute_metrics(extracted_str, original_str):
    extracted = parse_keywords(extracted_str)
    original  = parse_keywords(original_str)
    if not extracted or not original:
        return 0.0, 0.0, 0.0

    all_terms  = extracted + original
    embeddings = sem_model.encode(all_terms, convert_to_tensor=True)
    emb_ext    = embeddings[:len(extracted)]
    emb_orig   = embeddings[len(extracted):]

    matched_orig = set()
    tp = 0
    for i, e in enumerate(extracted):
        for j, o in enumerate(original):
            if j in matched_orig:
                continue
            string_match   = e in o or o in e
            cos_score      = st_util.cos_sim(emb_ext[i], emb_orig[j]).item()
            semantic_match = cos_score >= SIM_THRESHOLD
            if string_match or semantic_match:
                tp += 1
                matched_orig.add(j)
                break

    precision = tp / len(extracted)
    recall    = tp / len(original)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print("Calculez metrici semantice...")
results_df[['precision', 'recall', 'f1']] = results_df.apply(
    lambda r: pd.Series(compute_metrics(r['keywords_generated'], r['keywords_original'])),
    axis=1
)

avg_p  = results_df['precision'].mean()
avg_r  = results_df['recall'].mean()
avg_f1 = results_df['f1'].mean()

print(f"\n{'='*45}")
print(f"  Precision medie : {avg_p:.4f}  ({avg_p:.2%})")
print(f"  Recall mediu    : {avg_r:.4f}  ({avg_r:.2%})")
print(f"  F1 mediu        : {avg_f1:.4f}  ({avg_f1:.2%})")
print(f"{'='*45}")
print(f"  Documente cu F1 > 0 : {(results_df['f1'] > 0).sum()} / {len(results_df)}")


Se încarcă modelul semantic...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Calculez metrici semantice...

  Precision medie : 0.1716  (17.16%)
  Recall mediu    : 0.6682  (66.82%)
  F1 mediu        : 0.2650  (26.50%)
  Documente cu F1 > 0 : 164 / 168


In [13]:
N_PREVIEW = 20

def make_html_table(df_preview):
    rows = ''
    for _, r in df_preview.iterrows():
        color = '#c8f7c5' if r['f1'] >= 0.5 else '#fde8d8' if r['f1'] > 0 else '#f5f5f5'
        rows += f"""
        <tr>
            <td style='font-size:12px;color:#555;padding:6px 8px;'>{r['filename']}</td>
            <td style='color:#2980b9;font-weight:500;padding:6px 8px;'>{r['keywords_generated']}</td>
            <td style='color:#555;padding:6px 8px;'>{r['keywords_original']}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['precision']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['recall']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;background:{color};font-weight:bold;'>{r['f1']:.2f}</td>
        </tr>"""

    html = f"""
    <h3 style='font-family:sans-serif;'>🤖 RoGPT2 — Primele {N_PREVIEW} articole (set de test)</h3>
    <table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>
        <thead>
            <tr style='background:#2980b9;color:white;'>
                <th style='padding:8px;text-align:left;'>Fișier</th>
                <th style='padding:8px;text-align:left;'>Keywords generate (RoGPT2)</th>
                <th style='padding:8px;text-align:left;'>Keywords originale</th>
                <th style='padding:8px;text-align:center;'>Precision</th>
                <th style='padding:8px;text-align:center;'>Recall</th>
                <th style='padding:8px;text-align:center;'>F1</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style='font-family:sans-serif;font-size:12px;color:#888;margin-top:8px;'>
        🟢 F1 ≥ 0.5 &nbsp;|&nbsp; 🟠 F1 > 0 &nbsp;|&nbsp; ⚪ F1 = 0
    </p>
    """
    return html

display(HTML(make_html_table(results_df.head(N_PREVIEW))))

Fișier,Keywords generate (RoGPT2),Keywords originale,Precision,Recall,F1
RJP_2014_4_RO_Art-05.pdf,osmopenie; os; copil; adolescent; nou-născut; adult; vârstnic; vârstă; patologie; metabolism; vitamina d; calciu; paraproteinemie; hipercalciurie; hipopotasemie; deficit de vitamină D; hipospadias,osteopenie; copil; diagnostic; tratament,0.18,0.75,0.29
RJP_2008_3_Art-03.pdf,infecția congenitală; copil; nou-născut; sugar; mamă infectată; infecție; sarcină; anticorpi; sistem imunitar; imunitate; sindrom; virus; viral; parazitologie; vaccin; antigen; imunoglobuline; limfocite; celule imune;,sindromul torch; infecții congenitale perinatale,0.11,1.00,0.19
RJP_2008_2_Art-10.pdf,durere; membre inferioare; copil; vârstă; diagnostic; tratament; simptomatologie; evoluție; complicații; factori de risc pentru apariția durerii; metode de diagnostic și tratament ale durerii la copii; criterii de includere a copilului în programele de sănătate publică; recomandări de investigații și tratamente ale durerilor la copiii cu,dureri ale membrelor inferioare; copil,0.15,1.00,0.27
RMJ_2018_3_Art-07.pdf,hepatopatii virale; copil; evoluție; diagnostic; terapie antivirală; tablou clinic; simptomatologie; prognostic; tratament; complicații; factori de risc pentru infecția cu virus hepatitic C; virusuri hepatitice B și C (VHC); factori genetici și de,hepatită virală c; interferon; globule albe; hemoglobină,0.08,0.25,0.12
RJP_2017_2_RO_Art-06.pdf,malformație de canale aeriene; copil; malformație; pneumonie; obstrucție bronșică; insuficiență respiratorie acută; ventilație mecanică; bronhoscopie; diagnostic; tratament; evoluție; complicații; nou-născut; prematuritate; morbiditate și mortalitate crescute;,malformație congenitală pulmonară; chisturi; prognostic,0.20,1.00,0.33
RJP_2014_3_RO_Art-09.pdf,obezitate; copil; nou-născut; indice de masă corporală; risc de boală cardiovasculară; hipertensiune arterială; colesterol; aterom; hipercolesterolemie; diabet zaharat; boli de nutriție; copii; supraponderali; obezi; factori de risc cardiovascular; obezitate infantilă; sindrom metabol,obezitate; mic pentru vârsta gestaționalǎ; indicele intimă medie carotidian,0.12,0.67,0.20
RMJ_2018_3_Art-08.pdf,intoxicații acute; intoxicația acută voluntară; medicamente; copil; toxicomanie; alcoolism; tratament medicamentos; consum de băuturi alcoolice etilice; copii; terapie medicamentoasă; abuz de substanțe toxice; comă; deces; studii de caz; metaanalize; literatură de specialitate; reacții adverse,alcool etilic; substanțe de abuz; adolescent; medicamente,0.24,1.00,0.38
RJP_2017_4_RO_Art-03.pdf,"precoce, dar și un risc crescut de deces. ### Cuvinte cheie: insuficiența respiratororie; copil; prematuritate; deces; terapie intensivă; transfuzie de sânge; transplant de măduvă osoasă; anemie hemolitică acută; hemocromatoză; leucemie acută",insuficiență respiratorie; copil; detresă respiratorie acută,0.18,0.67,0.29
RJP_2006_3_Art-08.pdf,artropatie; copil; boală cronică; dizabilitate; distres; remisiune; recuperare; tratament; copii; reumatologie; pediatrie; boli reumatismale; artrite; distrofie musculară; osteoartrită; spondilită anchilozantă; reuma,artrita idiopatică juvenilă; etiologie și fiziopatologie; clasificare; diagnostic diferențial; tratament,0.18,0.60,0.27
RMJ_2019_1_Art-08.pdf,copil; limbaj verbal; performanță; teste; omeț; copii; copilarie; vorbire; personalitate; psihologie; psihoterapie; terapie comportamentală; psihodiagnostic; evaluare; diagnostic; tratament; dezvoltare personală; comunicare; relații interpersonale; familie; școală; societate; sănătate mintală;,limbaj verbal; terapie; monitorizare; test creion-hârtie,0.17,1.00,0.30
